In [1]:
import sys
from pathlib import Path
import pandas as pd

# Resolve project root (notebooks/ is 1 level below project root)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATASET_PATH

df = pd.read_csv(RAW_DATASET_PATH)

# 1. Check dimensions (must be 100,000 rows, 42 columns)
print("Shape:", df.shape)
assert df.shape == (100000, 42), f"Expected (100000, 42), got {df.shape}"

# 2. Check for missing values (must be 0 across all columns)
missing_count = df.isnull().sum().sum()
print("Total missing values:", missing_count)
assert missing_count == 0, f"Expected 0 missing values, got {missing_count}"

# 3. Print all column names and data types
print("\nActual Columns & Dtypes:")
for col, dtype in df.dtypes.items():
    print(f"  - {col}: {dtype}")


Shape: (100000, 42)
Total missing values: 0

Actual Columns & Dtypes:
  - record_id: int64
  - product_id: object
  - product_name: object
  - category: object
  - store_id: object
  - region: object
  - supplier_id: object
  - transaction_date: object
  - expiration_date: object
  - shelf_life_days: int64
  - days_remaining_at_purchase: int64
  - storage_temp: float64
  - temp_deviation: float64
  - base_price: float64
  - cost_price: float64
  - initial_quantity: int64
  - spoilage_sensitivity: float64
  - day_of_week: int64
  - is_weekend: int64
  - month: int64
  - daily_demand: int64
  - demand_variability: float64
  - temp_abuse_events: int64
  - distribution_hours: float64
  - handling_score: int64
  - packaging_score: int64
  - spoilage_risk: float64
  - was_spoiled: int64
  - quality_grade: object
  - days_until_expiry: int64
  - markdown_applied: int64
  - discount_pct: float64
  - selling_price: float64
  - units_sold: int64
  - units_wasted: int64
  - waste_pct: float64
  -

### Dataset Verification & Leakage Rule Log (M1-T01)

* **Dataset Status:** Successfully verified 100,000 rows and 42 columns with 0 missing values.
* **Schema Corrections Confirmed:**
  * `markdown` → `markdown_applied`
  * `demand_volatility` → `demand_variability`
  * `days_until` → `days_until_expiry`
  * `days_remaining` → `days_remaining_at_purchase`
  * `profit_margin` → `profit_margin_pct`
* **Feature Exclusion Notice (leakage rule):**
  * `spoilage_risk` is derived from the same process as the target `was_spoiled`. It is excluded as an input feature for both Track 3A and Track 3B and is used only as an evaluation benchmark. Track 3B takes Track 3A's out-of-fold predicted spoilage probability as its risk input instead.
  * Outcome columns (`units_sold`, `units_wasted`, `waste_pct`, `waste_cost`, `revenue`, `profit`, `profit_margin_pct`) are post-outcome values and are never used as Track 3A inputs. Where they serve as Track 3B targets, they are not inputs either.
* **New Columns Found During Verification:**
  * `spoilage_sensitivity` (needs the M1-T06 leakage check before use)
  * `temp_abuse_events`, `handling_score`, `packaging_score`, `supplier_score`
  * `is_promoted`
  * `initial_quantity`, `distribution_hours`
* **Notes for M1-T04:**
  * `transaction_date` and `expiration_date` are loaded as strings and need to be parsed to dates.
  * `day_of_week`, `is_weekend` and `month` already exist in the dataset.